# ROGII v45 - honest validation and anchored alignment

This revision keeps the sequence and geology feature families, but changes
how they are validated and deployed:

1. Formation columns are treated as training-only; both train and test use
   fold-local spatial imputations compatible with the hidden inference schema.
2. DTW is replaced by a last-known-TVT-anchored, free-end Viterbi alignment.
3. Spatial imputers are fitted inside each fold.
4. Blending, post-processing, and Savitzky-Golay smoothing are cross-fitted.
5. Spatial-block, causal-prefix, and feature-ablation diagnostics are available.
6. The production ensemble uses one LightGBM, CatBoost, and XGBoost model
   instead of nine highly correlated seed variants.

## 1. Setup

In [1]:
import os, time, subprocess, multiprocessing, warnings, zlib
import numpy as np
import pandas as pd
from pathlib import Path
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor, Pool
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from joblib import Parallel, delayed
from numba import njit
import lightgbm as lgb
import xgboost as xgb
import optuna
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK_RUN_VERSION = "ROGII_v45_honest_validation_2026_06_15"
_t0 = time.time()
def _dbg(msg): print(f"[{(time.time()-_t0)/60:.1f}m] {msg}", flush=True)
_dbg(f"=== {NOTEBOOK_RUN_VERSION} START ===")

RUNNING_ON_KAGGLE = os.path.exists("/kaggle/input")
_gpu_names = ""
try:
    _gpu_names = subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader",
        shell=True, stderr=subprocess.DEVNULL, timeout=20
    ).decode(errors="ignore").strip()
except Exception:
    pass
GPU_COUNT = len([x for x in _gpu_names.splitlines() if x.strip()])
CATBOOST_DEVICES = os.environ.get(
    "ROGII_CATBOOST_DEVICES",
    ":".join(str(i) for i in range(GPU_COUNT)) if GPU_COUNT else "0",
)
_dbg(f"GPUs: {_gpu_names!r} GPU_COUNT={GPU_COUNT}")
if GPU_COUNT == 0 and RUNNING_ON_KAGGLE:
    raise RuntimeError(
        "No GPU detected. Switch Accelerator to 'GPU T4 x2' in notebook settings.\n"
        "CPU runtime will exceed the 12h Kaggle timeout (v39 hit this exact failure)."
    )
if GPU_COUNT == 0:
    _dbg("WARN: no local GPU detected; model training will use CPU and may be slow")

SEED = 42
np.random.seed(SEED)
NCPU = min(4, multiprocessing.cpu_count())
LOCAL_DATASET_CANDIDATES = (
    Path("./data"),
    Path("./rogii-wellbore-geology-prediction"),
)

class CFG:
    if RUNNING_ON_KAGGLE:
        dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    else:
        dataset_path = next(
            (path for path in LOCAL_DATASET_CANDIDATES
             if (path / "train").is_dir() and (path / "test").is_dir()),
            LOCAL_DATASET_CANDIDATES[0],
        )
    seed = SEED
    n_splits = 5
    run_diagnostics = os.environ.get("ROGII_RUN_DIAGNOSTICS", "0") == "1"
    pp_trials = int(os.environ.get("ROGII_PP_TRIALS", "80"))
    diagnostic_estimators = int(os.environ.get("ROGII_DIAG_ESTIMATORS", "2500"))

if CFG.pp_trials < 1:
    raise ValueError("ROGII_PP_TRIALS must be at least 1")
if CFG.diagnostic_estimators < 1:
    raise ValueError("ROGII_DIAG_ESTIMATORS must be at least 1")
_dbg(f"Dataset path: {CFG.dataset_path.resolve()}")

FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10; DENSE_SPW = 60; DENSE_K = 20
N_SPLITS = CFG.n_splits

BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10,  8.0,  64.0, 2, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]

PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_INIT_V_STD = 0.02; PF_INIT_SPR = 0.5; PF_RESAMP = 0.5
PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3

ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001

DTW_RADII = (20, 50, 100, 200)
DTW_STOCH_K = 12
DTW_STOCH_TEMP = 3.0
DTW_MAX_STEP = 2
DTW_MOVE_PENALTY = 0.08
DTW_ANCHOR_PENALTY = 0.02

PF_NUM_SEEDS = 3

[0.0m] === ROGII_v45_honest_validation_2026_06_15 START ===
[0.0m] GPUs: 'Tesla T4\nTesla T4' GPU_COUNT=2
[0.0m] Dataset path: /kaggle/input/competitions/rogii-wellbore-geology-prediction


## 2. Numba kernels

In [2]:
@njit(cache=True)
def _seed_numba(seed):
    np.random.seed(seed)

@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i] * (1. - t) + grid[i + 1] * t

@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N + 1)
    for j in range(N): cum[j + 1] = cum[j] + w[j]
    u0 = np.random.uniform(0., 1. / N)
    np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0 + j / N
        while ci < N - 1 and cum[ci + 1] < u: ci += 1
        np2[j] = pos[ci] + rp * np.random.randn()
        na[j] = aux[ci] + rv * np.random.randn()
    return np2, na

@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30);     bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path

@njit(cache=True)
def _anchored_viterbi(query, ref, start_j, radius, max_step,
                      move_penalty, anchor_penalty, temperature):
    """Free-end alignment anchored near the last known TVT.

    Unlike endpoint-forced DTW, the path may finish anywhere in the local
    typewell window and may move slightly up or down in TVT.
    """
    N = len(query); M = len(ref); INF = 1e18
    j_lo = max(0, start_j - radius)
    j_hi = min(M - 1, start_j + radius)
    width = j_hi - j_lo + 1
    D = np.full((N, width), INF)
    parent = np.zeros((N, width), np.int8)
    for jj in range(width):
        j = j_lo + jj
        noise = 0.0
        if temperature > 0.0:
            u = np.random.uniform(1e-10, 1.0 - 1e-10)
            noise = -temperature * np.log(-np.log(u))
        D[0, jj] = ((query[0] - ref[j]) ** 2
                    + anchor_penalty * (j - start_j) ** 2 + noise)
    for i in range(1, N):
        for jj in range(width):
            j = j_lo + jj
            noise = 0.0
            if temperature > 0.0:
                u = np.random.uniform(1e-10, 1.0 - 1e-10)
                noise = -temperature * np.log(-np.log(u))
            local_cost = (query[i] - ref[j]) ** 2 + noise
            best = INF; best_d = 0
            for d in range(-max_step, max_step + 1):
                prev_jj = jj - d
                if prev_jj < 0 or prev_jj >= width:
                    continue
                trial = D[i - 1, prev_jj] + move_penalty * abs(d)
                if trial < best:
                    best = trial; best_d = d
            D[i, jj] = local_cost + best
            parent[i, jj] = np.int8(best_d)
    end_jj = 0
    for jj in range(1, width):
        if D[N - 1, jj] < D[N - 1, end_jj]:
            end_jj = jj
    path = np.empty(N, np.int64)
    path[N - 1] = end_jj
    for i in range(N - 1, 0, -1):
        path[i - 1] = path[i] - int(parent[i, path[i]])
    path += j_lo
    return path, D[N - 1, end_jj] / max(N, 1)

@njit(cache=True)
def _dtw_path_slope(path, smooth_win=5):
    N = len(path)
    slope = np.zeros(N, np.float32); hw = smooth_win // 2
    for i in range(N):
        i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
        if i1 > i0: slope[i] = float((path[i1] - path[i0]) / (i1 - i0))
        else: slope[i] = 1.0
    return slope

@njit(cache=True)
def _dtw_stochastic_realizations(query, ref, start_j, radius, K, temperature,
                                 max_step, move_penalty, anchor_penalty):
    N = len(query)
    paths = np.zeros((K, N), np.int64)
    for k in range(K):
        path, _ = _anchored_viterbi(
            query, ref, start_j, radius, max_step,
            move_penalty, anchor_penalty, temperature)
        paths[k] = path
    return paths

@njit(cache=True)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N,
             ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ls + IS * np.random.randn()
        rate[j] = ir + 0.01 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA * rate[j] + RN * np.random.randn()
            pos[j] += rate[j] * dm + PN * np.random.randn()
            tvt_j = pos[j] - z_v[i]
            tvt_j = max(tvt_j, vmin - 50.); tvt_j = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step)
                d = (gr_v[i] - eg) / gs
                lk = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300)
                w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1. / N
        tv = 0.
        for j in range(N): tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]
    return pts, std_

@njit(cache=True)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step,
          gs, ip, iv, beta, icpt, zsig, N,
          MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ip + 0.5 * np.random.randn()
        vel[j] = iv + 0.02 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.; pz = z_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        dzd = (z_v[i] - pz) / dm; ve = beta * dzd + icpt
        for j in range(N):
            vel[j] = MOM * vel[j] + VN * np.random.randn()
            pos[j] += vel[j] * dm + PN * np.random.randn()
            pos[j] = max(pos[j], vmin - 50.); pos[j] = min(pos[j], vmin + len(gg_p) * step + 50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step)
                dp = (gr_v[i] - ep) / gs
                lp = max(np.exp(-0.5 * dp * dp) if dp * dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step)
                    ds = (gr_sm_v[i] - es) / (gs * 1.5)
                    ls = max(np.exp(-0.5 * ds * ds) if ds * ds < 600. else 0., 1e-300)
                    lk = (1. - GR_WT) * lp + GR_WT * ls
                else:
                    lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j] - ve) / max(zsig * 2., 0.005)
            lz = max(np.exp(-0.5 * dv * dv) if dv * dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1. / N
        wm = 0.
        for j in range(N): wm += w[j] * pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j] * (pos[j] - wm) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_

## 3. Helper functions (multi-seed PF)

In [3]:
def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)

def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))

def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i

def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

def _standardize_signal(values):
    arr = np.asarray(values, np.float64).copy()
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr)
    arr[~finite] = np.median(arr[finite])
    return (arr - arr.mean()) / (arr.std() + 1e-6)

def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
    return tw_tvt[path].astype(np.float32)

def _pf_ancc_multi(hw, tw_tvt, tw_gr, N=ANCC_N, n_seeds=PF_NUM_SEEDS, base_seed=42):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts_stack = []; std_stack = []
    for s in range(n_seeds):
        _seed_numba(base_seed + s)
        pts, std = _pf_ancc(
            ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
            ev['GR'].values.astype(np.float64), gg, gmin, gst,
            gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN,
            ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
        pts_stack.append(pts.astype(np.float32))
        std_stack.append(std.astype(np.float32))
    return np.mean(pts_stack, 0).astype(np.float32), np.mean(std_stack, 0).astype(np.float32)

def _pf_z_multi(hw, tw_tvt, tw_gr, N=PF_N, n_seeds=PF_NUM_SEEDS, base_seed=42):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna['Z'].values); dvt = np.diff(kna['TVT_input'].values)
    dmd_k = np.diff(kna['MD'].values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2]; vt = dvt[m2] / dmd_k[m2]
        A = np.column_stack([vz, np.ones_like(vz)]); c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else:
        beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2['TVT_input'].values); dmd2 = np.diff(t2['MD'].values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts_stack = []; std_stack = []
    for s in range(n_seeds):
        _seed_numba(base_seed + 1000 + s)
        pts, std = _pf_z(
            ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
            ev['GR'].values.astype(np.float64),
            gr_sm.loc[ev.index].values.astype(np.float64),
            gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv,
            beta, icpt, zsig, N,
            PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
        pts_stack.append(pts.astype(np.float32))
        std_stack.append(std.astype(np.float32))
    return np.mean(pts_stack, 0).astype(np.float32), np.mean(std_stack, 0).astype(np.float32)

def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
    qn_f = _standardize_signal(query_gr)
    rn_f = _standardize_signal(tw_gr)
    start_j = _nn(tw_tvt, last_tvt)
    dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
    inv_cost_sum = 0.0; tvt_stack = []
    for r in radii:
        path, cost = _anchored_viterbi(
            qn_f, rn_f, start_j, r, DTW_MAX_STEP,
            DTW_MOVE_PENALTY, DTW_ANCHOR_PENALTY, 0.0)
        tvt_pred = tw_tvt[path].astype(np.float32)
        slope = _dtw_path_slope(path)
        dtw_tvts[r] = tvt_pred; dtw_slopes[r] = slope; dtw_costs[r] = cost
        ic = 1.0 / (cost + 1e-6); inv_cost_sum += ic
        tvt_stack.append((tvt_pred, ic))
    weights = np.array([ic / inv_cost_sum for _, ic in tvt_stack], dtype=np.float32)
    tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
    dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
    return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens

def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt,
                       radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP,
                       seed=SEED):
    N = len(query_gr)
    qn = _standardize_signal(query_gr)
    rn = _standardize_signal(tw_gr)
    start_j = _nn(tw_tvt, last_tvt)
    _seed_numba(seed)
    paths = _dtw_stochastic_realizations(
        qn, rn, start_j, radius, K, temperature, DTW_MAX_STEP,
        DTW_MOVE_PENALTY, DTW_ANCHOR_PENALTY)
    tvt_realiz = np.empty((K, N), dtype=np.float32)
    for k in range(K):
        for i in range(N): tvt_realiz[k, i] = tw_tvt[paths[k, i]]
    mean_tvt = tvt_realiz.mean(axis=0).astype(np.float32)
    std_tvt = tvt_realiz.std(axis=0).astype(np.float32)
    cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
    return mean_tvt, std_tvt, cv_tvt

# JIT warmup
_md = np.linspace(1, 50, 20, np.float64); _z = np.zeros(20, np.float64); _gr = np.full(20, 50., np.float64)
_gg = np.linspace(45, 55, 100, np.float64)
_pf_ancc(_md, _z, _gr, _gg, 45., 0.1, 20., 50., 0., 8, 0.998, 0.002, 0.005, 0.3, 0.1, 0.001, 0.5)
_pf_z(_md, _z, _gr, _gr, _gg, _gg, 45., 0.1, 20., 50., 0., -1., 0., 0.1, 8, 0.993, 0.005, 0.01, 0.3, 0.2, 0.003, 0.5)
_beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)
_q = np.random.randn(40); _r = np.random.randn(50)
_anchored_viterbi(_q, _r, 25, 10, 2, 0.08, 0.02, 0.0)
_dtw_stochastic_realizations(_q, _r, 25, 10, 3, 2.0, 2, 0.08, 0.02)
_dbg("JIT warmup done")

[0.3m] JIT warmup done


## 4. Calibration helpers, NCC

In [4]:
def robust_slope(x, y, w=None):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])

def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)

def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col; n = len(bv)
    b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n)); w /= w.sum()
    b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    sc_ens = (tvts * sw).sum(1).astype(np.float32)
    return out, sc_ens

## 5. Spatial imputers

In [5]:
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS)
            except Exception: continue
            numeric = df[['X', 'Y'] + FORMATIONS].apply(pd.to_numeric, errors='coerce')
            numeric[FORMATIONS] = numeric[FORMATIONS].replace(0, np.nan)
            xy = numeric[['X', 'Y']].dropna()
            formation_medians = numeric[FORMATIONS].median()
            if len(xy) == 0 or not np.isfinite(formation_medians.to_numpy()).all():
                continue
            row = {'wid': wid, 'x': float(xy['X'].median()), 'y': float(xy['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(formation_medians[c])
            rows.append(row)
        if not rows:
            raise ValueError("FormationPlaneKNN requires at least one well with formation surfaces")
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        xy_q = np.atleast_2d(xy_q)
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if nf == 1:
            dist = dist[:, None]; idx = idx[:, None]
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        nk = min(k, nf)
        ord_ = np.argpartition(dist, min(nk - 1, nf - 1), 1)[:, :nk]
        dk = np.take_along_axis(dist, ord_, 1); ik = np.take_along_axis(idx, ord_, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        xn = (self.xa[ik] - Xq[:, None]) / self.scale[0]
        yn = (self.ya[ik] - Yq[:, None]) / self.scale[1]
        fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1),
                        (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except Exception:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception: pass
        # Coordinates are centered on each query, so the local plane value at
        # the query is its intercept. This avoids raw-coordinate conditioning.
        pred = coef[:, 2, :].astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)

class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC'])
            except Exception: continue
            df = df.apply(pd.to_numeric, errors='coerce')
            df['ANCC'] = df['ANCC'].replace(0, np.nan)
            df = df.dropna()
            if len(df) == 0: continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid] * len(s))
        if not xs:
            raise ValueError("DenseANCCImputer requires at least one well with ANCC")
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if nf == 1:
            dist = dist[:, None]; idx = idx[:, None]
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        nk = min(k, nf)
        ord_ = np.argpartition(dist, min(nk - 1, nf - 1), 1)[:, :nk]
        dk = np.take_along_axis(dist, ord_, 1); ik = np.take_along_axis(idx, ord_, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32),                np.where(vk, dk, np.inf).min(1).astype(np.float32)

_dbg("Building FormationPlaneKNN + DenseANCCImputer")
hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
train_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]

# Public test examples can be recycled training wells, while the hidden test
# schema replaces training-only formation surfaces. Never infer availability
# from the public examples.
FORMATION_INPUT_COLUMNS = set()
_dbg("Direct formation columns disabled (training-only competition fields)")
FI = FormationPlaneKNN(train_wids, CFG.dataset_path / "train")
DI = DenseANCCImputer(train_wids, CFG.dataset_path / "train")
_FI = FI; _DI = DI
_dbg(f"FI rows: {len(FI.df)}; DI rows: {len(DI.ancc):,d}")

[0.3m] Building FormationPlaneKNN + DenseANCCImputer
[0.3m] Direct formation columns disabled (training-only competition fields)
[0.9m] FI rows: 765; DI rows: 45,960


## 6. Build well features (multi-seed PF)

In [6]:
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
DTW_OFFS  = np.array([-20, -10, -5, -2, 0, 2, 5, 10, 20], np.float32)

def _stable_well_seed(wid):
    return SEED + int(zlib.crc32(wid.encode('utf-8')) & 0x7FFFFFFF)

def _prepare_typewell(tw, wid):
    missing = {'TVT', 'GR'} - set(tw.columns)
    if missing:
        raise ValueError(f"{wid}: typewell missing columns {sorted(missing)}")
    clean = tw[['TVT', 'GR']].apply(pd.to_numeric, errors='coerce').dropna()
    clean = clean.groupby('TVT', as_index=False, sort=True)['GR'].mean()
    if len(clean) < 3 or not np.isfinite(clean[['TVT', 'GR']].to_numpy()).all():
        raise ValueError(f"{wid}: typewell requires at least three finite TVT/GR rows")
    if np.any(np.diff(clean['TVT'].to_numpy()) <= 0):
        raise ValueError(f"{wid}: typewell TVT values must be strictly increasing")
    return clean

def _split_known_eval(hw, wid):
    required = {'MD', 'Z', 'GR', 'X', 'Y', 'TVT_input'}
    missing = required - set(hw.columns)
    if missing:
        raise ValueError(f"{wid}: horizontal well missing columns {sorted(missing)}")
    hw = hw.reset_index(drop=True).copy()
    for col in required:
        hw[col] = pd.to_numeric(hw[col], errors='coerce')
    if 'TVT' in hw.columns:
        hw['TVT'] = pd.to_numeric(hw['TVT'], errors='coerce')
    if hw['MD'].isna().any() or np.any(np.diff(hw['MD'].to_numpy()) < 0):
        raise ValueError(f"{wid}: MD must be finite and nondecreasing")
    if hw[['Z', 'X', 'Y']].isna().any().any():
        raise ValueError(f"{wid}: Z, X, and Y must be finite")
    known = hw['TVT_input'].notna().to_numpy()
    hidden_idx = np.flatnonzero(~known)
    if len(hidden_idx) == 0:
        raise ValueError(f"{wid}: no hidden TVT suffix found")
    split_at = int(hidden_idx[0])
    if split_at < 10 or not known[:split_at].all() or known[split_at:].any():
        raise ValueError(
            f"{wid}: TVT_input must contain a contiguous known prefix of at least 10 rows "
            "followed by a contiguous hidden suffix")
    return hw, hw.iloc[:split_at].copy(), hw.iloc[split_at:].copy()

def build_well(hw_path, tw_path, is_train):
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path)
        tw = _prepare_typewell(pd.read_csv(tw_path), wid)
        hw, kn, ev = _split_known_eval(hw, wid)
    except Exception as exc:
        raise RuntimeError(f"Failed to prepare well {wid}: {exc}") from exc
    if is_train and 'TVT' not in hw.columns:
        raise RuntimeError(f"{wid}: training horizontal well is missing TVT")
    if is_train and ev['TVT'].isna().any():
        raise RuntimeError(f"{wid}: training targets must be finite throughout the hidden suffix")
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    well_seed = _stable_well_seed(wid)

    pf_a, std_a = _pf_ancc_multi(
        hw, tw_tvt, tw_gr, n_seeds=PF_NUM_SEEDS, base_seed=well_seed)
    if len(pf_a) == 0:
        raise RuntimeError(f"{wid}: ANCC particle filter produced no predictions")
    pf_z_a, std_z = _pf_z_multi(
        hw, tw_tvt, tw_gr, n_seeds=PF_NUM_SEEDS, base_seed=well_seed)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z_a) == len(pf_a) and not np.any(np.isnan(pf_z_a))

    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[len(kn):].to_numpy(np.float32)
    kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.

    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]
    sc_cons = (sc8 + sc15 + sc25) / 3.
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens

    # Align only the hidden continuation. The final typewell endpoint is free;
    # the path is anchored at the last observed TVT.
    dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
        hgr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII)
    dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
        hgr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K,
        temperature=DTW_STOCH_TEMP, seed=well_seed + 2000)
    nh = len(ev)
    dtw_ens_ev = dtw_ens_ms.astype(np.float32)
    dtw_mean_ev = dtw_mean_stoch.astype(np.float32)
    dtw_std_ev = dtw_std_stoch.astype(np.float32)
    dtw_cv_ev = dtw_cv_stoch.astype(np.float32)
    dtw_per_radius_ev = {}; dtw_slope_ev = {}
    for r in DTW_RADII:
        dtw_per_radius_ev[r] = dtw_tvts_ms[r].astype(np.float32)
        dtw_slope_ev[r] = dtw_slopes_ms[r].astype(np.float32)
    dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
    dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
    dtw_cost_min = float(dtw_cost_arr.min())
    dtw_cost_range = float(dtw_cost_arr.max() - dtw_cost_arr.min())

    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn['MD'].to_numpy(np.float32); kz = kn['Z'].to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:])
    slp_z = robust_slope(kz, ktvt)

    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64); xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32); z_ev = ev['Z'].to_numpy(np.float32)
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f; tvt_fs[f'tvtFw_{fn}'] = tvt_fw
        tvt_fs[f'tvtF50_{fn}'] = tvt_f50
        tvt_fs[f'bw_{fn}'] = np.float32(b_full); tvt_fs[f'bww_{fn}'] = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2)))
        form_list.append(tvt_f)
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d = fs.std(1).astype(np.float32)
    form_rng_d = (fs.max(1) - fs.min(1)).astype(np.float32)

    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid)
    d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    b_vd = ktvt + z_kn - d_kn
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_d = float(np.median(b_vd))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn
    d_rmse = float(np.sqrt(np.mean(res_kn ** 2)))
    d_bias = float(np.mean(res_kn)); d_nb_std = float(np.mean(d_std_kn))

    # Hidden test files may omit the training-only formation columns. Start
    # from spatially imputed surfaces and replace them with observed values
    # only where the current file actually supplies them.
    fm_ev = form_ev.astype(np.float64).copy()
    for fi, fn in enumerate(FORMATIONS):
        if fn not in FORMATION_INPUT_COLUMNS or fn not in ev.columns:
            continue
        observed = pd.to_numeric(ev[fn], errors="coerce").to_numpy(np.float64)
        valid = np.isfinite(observed)
        fm_ev[valid, fi] = observed[valid]
    # 1. fmd_<fm>: signed distance formation_depth - z_row (in TVT units of Z)
    fmd_feats = {}
    for fi, fn in enumerate(FORMATIONS):
        fmd_feats[f'fmd_{fn}'] = (fm_ev[:, fi] - z_ev.astype(np.float64)).astype(np.float32)
    # 2. thk_<fm1>_<fm2>: thickness between adjacent formation pairs
    thk_feats = {}
    for fi in range(len(FORMATIONS) - 1):
        a, b = FORMATIONS[fi], FORMATIONS[fi + 1]
        thk_feats[f'thk_{a}_{b}'] = (fm_ev[:, fi + 1] - fm_ev[:, fi]).astype(np.float32)
    # 3. pos_strat: fractional position within local stratigraphic column
    fm_min = fm_ev.min(axis=1)
    fm_max = fm_ev.max(axis=1)
    col_thk = np.maximum(fm_max - fm_min, 1.0)  # avoid div-by-zero
    pos_strat = ((z_ev.astype(np.float64) - fm_min) / col_thk).astype(np.float32)
    # 4. Closest formation index (0..5) and signed distance to it
    abs_fmd = np.abs(np.stack([fmd_feats[f'fmd_{fn}'] for fn in FORMATIONS], axis=1))
    closest_fm_idx = np.argmin(abs_fmd, axis=1).astype(np.float32)
    closest_fm_dist = np.array([fmd_feats[f'fmd_{FORMATIONS[ci]}'][i]
                                 for i, ci in enumerate(closest_fm_idx.astype(int))],
                                dtype=np.float32)

    all_sigs = [pf_use] + [p for p in bpaths.values()] +                [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense, dtw_ens_ev]
    sig_mat = np.stack(all_sigs, 1)
    sig_std = sig_mat.std(1).astype(np.float32)
    sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)

    gr_s = pd.Series(gr_full.values); rolls = {}
    for w in [5, 21, 51, 101]:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f'grm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}'] = gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2 = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum((gr_s ** 2).rolling(21, center=True, min_periods=1).mean(), 0.)
                     ).iloc[ev.index].values.astype(np.float32)

    # Strictly trailing GR features for the operational/causal diagnostic.
    cgr_s = hw['GR'].astype(float).ffill().fillna(float(np.nanmean(tw_gr)))
    causal_rolls = {}
    for w in [5, 21, 51]:
        r = cgr_s.rolling(w, min_periods=1)
        causal_rolls[f'cgrm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32)
        causal_rolls[f'cgrs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        causal_rolls[f'cglag{lag}'] = (
            cgr_s.shift(lag).ffill().iloc[ev.index].values.astype(np.float32))
    cgr = cgr_s.iloc[ev.index].values.astype(np.float32)
    cgr_d1 = cgr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    cgr_d2 = cgr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)

    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32)
    slp_b_50 = (last_tvt + slp_50 * md_since).astype(np.float32)
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
    def sc(v): return np.full(nh, np.float32(v), np.float32)

    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt': sc(last_tvt),
        'pf_ancc': pf_use, 'pf_ancc_std': std_use,
        'pf_ancc_delta': (pf_use - last_tvt).astype(np.float32),
        'pf_z': (pf_z_a.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta': ((pf_z_a - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z': ((pf_use - pf_z_a.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        'sc8_d': (sc8 - np.float32(last_tvt)).astype(np.float32), 'sc8_sc': sc8s,
        'sc15_d': (sc15 - np.float32(last_tvt)).astype(np.float32), 'sc15_sc': sc15s,
        'sc25_d': (sc25 - np.float32(last_tvt)).astype(np.float32), 'sc25_sc': sc25s,
        'sc_cons_d': (sc_cons - np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d': (sc_ens - np.float32(last_tvt)).astype(np.float32),
        'sc_trust': sc(sc_trust), 'hyb_d': (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std': sig_std, 'sig_mean_d': sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}': sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d': form_mean_d, 'form_std_d': form_std_d, 'form_rng_d': form_rng_d,
        'spatial_ancc_d': (form_ev[:, 0] - z_ev).astype(np.float32),
        'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32),
        'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32),
        'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': sc(d_rmse), 'dense_bias': sc(d_bias), 'dense_nb_std': sc(d_nb_std),
        'pf_vs_spatial': (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense': (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense': (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32),
        'beam_vs_spatial': (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam': (sc_ens - bpaths['cons']).astype(np.float32),
        'dtw_ens_d': (dtw_ens_ev - last_tvt).astype(np.float32),
        'dtw_stoch_mean_d': (dtw_mean_ev - last_tvt).astype(np.float32),
        'dtw_stoch_std': dtw_std_ev,
        'dtw_stoch_cv': dtw_cv_ev,
        'dtw_slope_mean': dtw_slope_mean_ev,
        **{f'dtw_r{r}_d': (dtw_per_radius_ev[r] - last_tvt).astype(np.float32) for r in DTW_RADII},
        **{f'dtw_slope_r{r}': dtw_slope_ev[r] for r in DTW_RADII},
        'dtw_cost_min': sc(dtw_cost_min),
        'dtw_cost_range': sc(dtw_cost_range),
        'dtw_vs_beam': (dtw_ens_ev - bpaths['cons']).astype(np.float32),
        'dtw_vs_pf': (dtw_ens_ev - pf_use).astype(np.float32),
        'dtw_vs_sc': (dtw_ens_ev - sc_ens).astype(np.float32),
        **{f'tddtw{int(o)}': hgr - np.interp(dtw_ens_ev + o, tw_tvt, tw_gr).astype(np.float32)
           for o in DTW_OFFS},
        'cal_a': sc(a_cal), 'cal_b': sc(b_cal),
        'pfx_rmse': sc(pfx_rmse), 'known_len': sc(len(kn)), 'eval_len': sc(nh),
        'slp_all': sc(slp_all), 'slp_50': sc(slp_50), 'slp_z': sc(slp_z),
        'slp_b_d_all': (slp_b_all - last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 - last_tvt).astype(np.float32),
        'ktvt_range': sc(float(np.ptp(ktvt))), 'ktvt_std': sc(float(ktvt.std())),
        'md_since': md_since, 'frac': frac, 'frac2': frac ** 2, 'sqrt_frac': np.sqrt(frac),
        'z': z_ev,
        'dx': (ev['X'] - float(lk['X'])).to_numpy(np.float32),
        'dy': (ev['Y'] - float(lk['Y'])).to_numpy(np.float32),
        'dz': (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy': np.sqrt((ev['X'] - float(lk['X'])) ** 2 + (ev['Y'] - float(lk['Y'])) ** 2).to_numpy(np.float32),
        'dzdmd': dzdmd, 'dxdmd': dxdmd, 'dydmd': dydmd,
        'gr': hgr, 'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env, 'gr_nrg': gr_nrg,
        'cgr': cgr, 'cgr_d1': cgr_d1, 'cgr_d2': cgr_d2,
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'gr_vs_slp_all': hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}': hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}': hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}': hgr - np.interp(sc_ens + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}': hgr - np.float32(np.interp(pf_use + o, tw_tvt, tw_gr)) for o in PF_OFFS},
        'tw_range': sc(float(np.ptp(tw_tvt))), 'tw_gr_mean': sc(float(tw_gr.mean())),
        # Formation-anchored features:
        **fmd_feats,
        **thk_feats,
        'pos_strat': pos_strat,
        'closest_fm_idx': closest_fm_idx,
        'closest_fm_dist': closest_fm_dist,
    }
    for k, v in rolls.items(): feats[k] = v
    for k, v in causal_rolls.items(): feats[k] = v
    result = pd.DataFrame(feats)
    if is_train:
        result['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
    return result

def build_dataset(paths, is_train, label):
    args = []
    missing_typewells = []
    for p in paths:
        wid = p.stem.replace('__horizontal_well', '')
        tw_path = p.parent / f'{wid}__typewell.csv'
        if tw_path.exists():
            args.append((wid, str(p), str(tw_path), is_train))
        else:
            missing_typewells.append(wid)
    if missing_typewells:
        message = f"build_dataset[{label}] missing typewells: {missing_typewells[:10]}"
        raise RuntimeError(message)
    _dbg(f"build_dataset[{label}]: {len(args)} wells, NCPU={NCPU}")
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=0)(
        delayed(build_well)(hp, tp, it) for _, hp, tp, it in args)
    failed_wells = [args[i][0] for i, result in enumerate(res) if result is None]
    if failed_wells:
        message = f"build_dataset[{label}] failed wells: {failed_wells[:10]}"
        raise RuntimeError(message)
    parts = [r for r in res if r is not None]
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if out.empty:
        raise RuntimeError(f"build_dataset[{label}] produced no rows")
    if out['id'].duplicated().any():
        duplicate_ids = out.loc[out['id'].duplicated(), 'id'].head(10).tolist()
        raise RuntimeError(f"build_dataset[{label}] duplicate IDs: {duplicate_ids}")
    numeric_columns = [c for c in out.columns if c not in {'well', 'id'}]
    out[numeric_columns] = out[numeric_columns].replace([np.inf, -np.inf], np.nan)
    if is_train and out['target'].isna().any():
        raise RuntimeError(f"build_dataset[{label}] contains non-finite targets")
    _dbg(f"build_dataset[{label}]: built {len(out):,d} rows, {len(out.columns)} cols")
    return out

SPATIAL_CORE_COLUMNS = [
    *[f'{prefix}_{fn}' for fn in FORMATIONS
      for prefix in ('tvtF', 'tvtFw', 'tvtF50', 'bw', 'bww', 'bw50',
                     'bw_early', 'bw_mid', 'frm_rmse')],
    'form_mean_d', 'form_std_d', 'form_rng_d',
    'spatial_ancc_d', 'spatial_knn_dist',
    'dense_ancc', 'dense_std', 'dense_dist',
    'tvt_dense_d', 'tvt_densew_d', 'tvt_dense50_d',
    'dense_rmse', 'dense_bias', 'dense_nb_std',
    *[f'fmd_{fn}' for fn in FORMATIONS],
    *[f'thk_{FORMATIONS[i]}_{FORMATIONS[i + 1]}' for i in range(len(FORMATIONS) - 1)],
    'pos_strat', 'closest_fm_idx', 'closest_fm_dist',
]
SPATIAL_DERIVED_COLUMNS = [
    'pf_vs_spatial', 'pf_vs_dense', 'spatial_vs_dense',
    'beam_vs_spatial', 'sig_std', 'sig_mean_d',
]
SPATIAL_FEATURE_COLUMNS = set(SPATIAL_CORE_COLUMNS + SPATIAL_DERIVED_COLUMNS)

def build_spatial_well(hw_path, is_train, fi, di):
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path)
        hw, kn, ev = _split_known_eval(hw, wid)
    except Exception as exc:
        raise RuntimeError(f"Failed to prepare spatial well {wid}: {exc}") from exc
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64)
    xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = fi.impute(xy_ev, self_wid=swid)
    form_kn, _ = fi.impute(xy_kn, self_wid=swid)
    d_ancc, d_std, d_dist = di.impute(xy_ev, self_wid=swid)
    d_kn, d_std_kn, _ = di.impute(xy_kn, self_wid=swid)

    ktvt = kn['TVT_input'].to_numpy(np.float32)
    z_kn = kn['Z'].to_numpy(np.float32)
    z_ev = ev['Z'].to_numpy(np.float32)
    last_tvt = float(ktvt[-1])
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(
            ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f
        tvt_fs[f'tvtFw_{fn}'] = tvt_fw
        tvt_fs[f'tvtF50_{fn}'] = tvt_f50
        tvt_fs[f'bw_{fn}'] = np.full(len(ev), b_full, np.float32)
        tvt_fs[f'bww_{fn}'] = np.full(len(ev), b_wls, np.float32)
        tvt_fs[f'bw50_{fn}'] = np.full(len(ev), b_late, np.float32)
        tvt_fs[f'bw_early_{fn}'] = np.full(len(ev), b_early, np.float32)
        tvt_fs[f'bw_mid_{fn}'] = np.full(len(ev), b_mid, np.float32)
        rmse = np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2))
        form_rmse[fn] = np.full(len(ev), rmse, np.float32)
        form_list.append(tvt_f)
    fs = np.stack(form_list, 1)

    _, _, _, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_vd = ktvt + z_kn - d_kn
    b_d = float(np.median(b_vd))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn

    fm_ev = form_ev.astype(np.float64).copy()
    for col_i, fn in enumerate(FORMATIONS):
        if fn not in FORMATION_INPUT_COLUMNS or fn not in ev.columns:
            continue
        observed = pd.to_numeric(ev[fn], errors='coerce').to_numpy(np.float64)
        valid = np.isfinite(observed)
        fm_ev[valid, col_i] = observed[valid]
    fmd = fm_ev - z_ev.astype(np.float64)[:, None]
    fm_min = fm_ev.min(axis=1); fm_max = fm_ev.max(axis=1)
    col_thk = np.maximum(fm_max - fm_min, 1.0)
    closest_idx = np.argmin(np.abs(fmd), axis=1)

    feats = {
        'id': [f'{wid}_{i}' for i in ev.index],
        **tvt_fs,
        **{f'frm_rmse_{fn}': form_rmse[fn] for fn in FORMATIONS},
        'form_mean_d': (fs.mean(1) - last_tvt).astype(np.float32),
        'form_std_d': fs.std(1).astype(np.float32),
        'form_rng_d': (fs.max(1) - fs.min(1)).astype(np.float32),
        'spatial_ancc_d': (form_ev[:, 0] - z_ev).astype(np.float32),
        'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32),
        'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32),
        'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': np.full(len(ev), np.sqrt(np.mean(res_kn ** 2)), np.float32),
        'dense_bias': np.full(len(ev), np.mean(res_kn), np.float32),
        'dense_nb_std': np.full(len(ev), np.mean(d_std_kn), np.float32),
        **{f'fmd_{fn}': fmd[:, i].astype(np.float32)
           for i, fn in enumerate(FORMATIONS)},
        **{f'thk_{FORMATIONS[i]}_{FORMATIONS[i + 1]}':
           (fm_ev[:, i + 1] - fm_ev[:, i]).astype(np.float32)
           for i in range(len(FORMATIONS) - 1)},
        'pos_strat': ((z_ev - fm_min) / col_thk).astype(np.float32),
        'closest_fm_idx': closest_idx.astype(np.float32),
        'closest_fm_dist': fmd[np.arange(len(ev)), closest_idx].astype(np.float32),
    }
    return pd.DataFrame(feats)

def build_spatial_dataset(paths, is_train, label, fi, di):
    _dbg(f"build_spatial_dataset[{label}]: {len(paths)} wells")
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=0)(
        delayed(build_spatial_well)(str(p), is_train, fi, di) for p in paths)
    parts = [r for r in res if r is not None]
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if out.empty:
        raise RuntimeError(f"build_spatial_dataset[{label}] produced no rows")
    if out['id'].duplicated().any():
        duplicate_ids = out.loc[out['id'].duplicated(), 'id'].head(10).tolist()
        raise RuntimeError(
            f"build_spatial_dataset[{label}] duplicate IDs: {duplicate_ids}")
    numeric_columns = [c for c in out.columns if c != 'id']
    values = out[numeric_columns].to_numpy(np.float64)
    if not np.isfinite(values).all():
        bad_columns = out[numeric_columns].columns[
            ~np.isfinite(values).all(axis=0)].tolist()
        raise RuntimeError(
            f"build_spatial_dataset[{label}] has non-finite columns: {bad_columns[:10]}")
    _dbg(f"build_spatial_dataset[{label}]: built {len(out):,d} rows")
    return out

def overlay_spatial_features(base_df, spatial_df, feature_names):
    X_fold = base_df[feature_names].copy()
    missing_columns = [
        col for col in SPATIAL_CORE_COLUMNS
        if col in X_fold.columns and col not in spatial_df.columns
    ]
    if missing_columns:
        raise RuntimeError(
            f"Fold-local spatial dataset is missing columns: {missing_columns[:5]}")
    aligned = spatial_df.set_index('id').reindex(base_df['id'])
    if aligned.index.has_duplicates or aligned.isnull().all(axis=1).any():
        raise RuntimeError("Fold-local spatial features do not align with base rows")
    for col in SPATIAL_CORE_COLUMNS:
        if col in X_fold.columns and col in aligned.columns:
            X_fold[col] = aligned[col].to_numpy()

    last = X_fold['last_known_tvt'].to_numpy()
    pf = X_fold['pf_ancc'].to_numpy()
    tvt_f = X_fold['tvtF_ANCC'].to_numpy()
    tvt_dense = last + X_fold['tvt_dense_d'].to_numpy()
    derived = {
        'pf_vs_spatial': pf - tvt_f,
        'pf_vs_dense': pf - tvt_dense,
        'spatial_vs_dense': tvt_f - tvt_dense,
        'beam_vs_spatial': last + X_fold['beam_cons_d'].to_numpy() - tvt_f,
    }
    sigs = [pf]
    sigs.extend(last + X_fold[f'beam_{tag}_d'].to_numpy() for *_, tag in BEAMS)
    sigs.extend([
        last + X_fold['sc8_d'].to_numpy(),
        last + X_fold['sc15_d'].to_numpy(),
        last + X_fold['sc25_d'].to_numpy(),
        last + X_fold['sc_ens_d'].to_numpy(),
        tvt_f, tvt_dense,
        last + X_fold['dtw_ens_d'].to_numpy(),
    ])
    sig_mat = np.stack(sigs, axis=1)
    derived['sig_std'] = sig_mat.std(1).astype(np.float32)
    derived['sig_mean_d'] = (sig_mat.mean(1) - last).astype(np.float32)
    for col, values in derived.items():
        if col in X_fold.columns:
            X_fold[col] = values
    return X_fold

## 7. Build matrices

In [ ]:
_dbg(f"Building train features ({PF_NUM_SEEDS}x PF seeds)")
train_df = build_dataset(hw_paths, is_train=True, label="train")
_dbg("Building test features")
test_df = build_dataset(test_paths, is_train=False, label="test")

train_features = [c for c in train_df.columns if c not in {'well', 'id', 'target'}]
test_features = [c for c in test_df.columns if c not in {'well', 'id', 'target'}]
missing_test_features = sorted(set(train_features) - set(test_features))
extra_test_features = sorted(set(test_features) - set(train_features))
if missing_test_features or extra_test_features:
    raise RuntimeError(
        "Train/test feature schema mismatch: "
        f"missing_test={missing_test_features[:10]}, "
        f"extra_test={extra_test_features[:10]}")
features = train_features
y = train_df['target']
_dbg(f"Features: {len(features)} | train rows: {len(train_df):,d} | test rows: {len(test_df):,d}")

[0.9m] Building train features (3x PF seeds)
[1.0m] build_dataset[train]: 773 wells, NCPU=4


## 8. Stratified group K-fold

In [ ]:
def stratified_group_kfold(groups, strat_keys, n_splits=5, seed=42):
    df = pd.DataFrame({'g': groups, 's': strat_keys})
    well_strat = df.groupby('g')['s'].mean().reset_index()
    well_strat = well_strat.sort_values('s').reset_index(drop=True)
    rng = np.random.RandomState(seed)
    n_w = len(well_strat)
    if n_w < n_splits:
        raise ValueError(
            f"Need at least {n_splits} wells for grouped validation; found {n_w}")
    chunk_size = max(1, n_splits)
    well_to_fold = {}
    for chunk_start in range(0, n_w, chunk_size):
        chunk = well_strat.iloc[chunk_start:chunk_start + chunk_size]
        perm = rng.permutation(len(chunk))
        for i, idx in enumerate(perm):
            well_to_fold[chunk['g'].iloc[idx]] = i % n_splits
    fold_of_row = df['g'].map(well_to_fold).values
    splits = []
    for k in range(n_splits):
        va = np.where(fold_of_row == k)[0]
        tr = np.where(fold_of_row != k)[0]
        splits.append((tr, va))
    return splits

def spatial_block_kfold(paths, row_groups, n_splits=5, seed=42):
    rows = []
    for path in paths:
        wid = path.stem.replace('__horizontal_well', '')
        try:
            xy = pd.read_csv(path, usecols=['X', 'Y'])
        except Exception:
            continue
        x = float(pd.to_numeric(xy['X'], errors='coerce').median())
        y_coord = float(pd.to_numeric(xy['Y'], errors='coerce').median())
        if np.isfinite(x) and np.isfinite(y_coord):
            rows.append((wid, x, y_coord))
    wells = pd.DataFrame(rows, columns=['well', 'x', 'y'])
    if len(wells) < n_splits:
        raise ValueError("Not enough wells for spatially blocked validation")
    coords = wells[['x', 'y']].to_numpy(np.float64)
    scale = np.where(coords.std(0) < 1e-6, 1.0, coords.std(0))
    labels = KMeans(n_clusters=n_splits, random_state=seed, n_init=20).fit_predict(
        (coords - coords.mean(0)) / scale)
    well_to_fold = dict(zip(wells['well'], labels))
    fold_series = pd.Series(row_groups).map(well_to_fold)
    if fold_series.isna().any():
        missing_wells = sorted(pd.Series(row_groups)[fold_series.isna()].unique())
        raise RuntimeError(
            f"Spatial validation could not map wells: {missing_wells[:10]}")
    fold_of_row = fold_series.to_numpy()
    return [
        (np.where(fold_of_row != fold)[0], np.where(fold_of_row == fold)[0])
        for fold in range(n_splits)
    ]

def validate_cv_splits(cv_splits, n_rows, name):
    if len(cv_splits) != N_SPLITS:
        raise RuntimeError(
            f"{name}: expected {N_SPLITS} folds, found {len(cv_splits)}")
    validation_counts = np.zeros(n_rows, np.int16)
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        if len(tr_idx) == 0 or len(va_idx) == 0:
            raise RuntimeError(
                f"{name}: fold {fold} has {len(tr_idx)} train and {len(va_idx)} validation rows")
        if np.intersect1d(tr_idx, va_idx).size:
            raise RuntimeError(f"{name}: fold {fold} has overlapping train/validation rows")
        validation_counts[va_idx] += 1
    invalid = np.flatnonzero(validation_counts != 1)
    if len(invalid):
        raise RuntimeError(
            f"{name}: {len(invalid)} rows do not appear in exactly one validation fold; "
            f"examples={invalid[:10].tolist()}")

def build_fold_spatial_overlays(cv_splits, label, include_test=True):
    train_overlays = []; test_overlays = []
    for fold, (tr_idx, _) in enumerate(cv_splits):
        fold_wells = sorted(train_df.iloc[tr_idx]['well'].unique())
        _dbg(f"{label} fold {fold}: fitting spatial imputers on {len(fold_wells)} wells")
        fold_fi = FormationPlaneKNN(fold_wells, CFG.dataset_path / "train")
        fold_di = DenseANCCImputer(fold_wells, CFG.dataset_path / "train")
        train_overlays.append(build_spatial_dataset(
            hw_paths, True, f"{label}-train-{fold}", fold_fi, fold_di))
        if include_test:
            test_overlays.append(build_spatial_dataset(
                test_paths, False, f"{label}-test-{fold}", fold_fi, fold_di))
    return train_overlays, test_overlays

strat_keys = train_df['eval_len'].values
splits = stratified_group_kfold(train_df['well'].values, strat_keys, n_splits=N_SPLITS, seed=SEED)
validate_cv_splits(splits, len(train_df), "group CV")
for k, (tr, va) in enumerate(splits):
    val_wells = train_df.iloc[va]['well'].nunique()
    val_eval_mean = train_df.iloc[va]['eval_len'].mean()
    _dbg(f"  fold {k}: {len(va):,d} val rows, {val_wells} val wells, "
         f"mean eval_len={val_eval_mean:.0f}")

_dbg("Building fold-local spatial features for honest CV")
fold_spatial_train, fold_spatial_test = build_fold_spatial_overlays(splits, "group")

spatial_splits = None
spatial_diag_overlays = None
if CFG.run_diagnostics:
    spatial_splits = spatial_block_kfold(
        hw_paths, train_df['well'].values, n_splits=N_SPLITS, seed=SEED)
    validate_cv_splits(spatial_splits, len(train_df), "spatial CV")
    for k, (_, va) in enumerate(spatial_splits):
        _dbg(f"  spatial fold {k}: {len(va):,d} rows, "
             f"{train_df.iloc[va]['well'].nunique()} wells")
    spatial_diag_overlays, _ = build_fold_spatial_overlays(
        spatial_splits, "spatial", include_test=False)

## 9. Model training - three diverse learners

Same regularization as v39:
- LGB: `min_child_samples=20`, `reg_lambda=3.5`
- CB:  `min_data_in_leaf=20`, `l2_leaf_reg=3.0`
- XGB: `min_child_weight=6`, `reg_lambda=3.0`

**Removed:** 4th LGB seed at `lr=0.015` (v39 showed it added <0.05 RMSE diversity
but ~25% extra runtime). **Trimmed:** CB/XGB iterations 10000 to 8000 (early stop
at 400 fires well before 8000 on most folds).

In [ ]:
# One configuration per learner. Seed variants were highly correlated and
# consumed most of the runtime without adding a distinct signal source.
lgb_params_base = dict(
    boosting_type="gbdt",
    num_leaves=255,
    min_child_samples=20,      # v36=15, v38=30, v39/v40=20
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=3.5,            # v36=3.0, v38=4.0, v39/v40=3.5
    reg_alpha=0.05,
    objective="regression", verbose=-1, n_jobs=-1,
    device_type="gpu" if GPU_COUNT > 0 else "cpu",
    gpu_use_dp=False, max_bin=255,
)
lgb_params = [
    dict(learning_rate=0.025, n_estimators=7000, seed=42, **lgb_params_base),
]

cb_params_base = dict(
    iterations=8000, depth=7,        # v39=10000, v40=8000
    l2_leaf_reg=3.0,                 # v36=2.0, v38=4.0, v39/v40=3.0
    min_data_in_leaf=20,             # v36=15, v38=30, v39/v40=20
    border_count=254,
    loss_function="RMSE",
    task_type="GPU" if GPU_COUNT > 0 else "CPU",
    od_type="Iter", od_wait=400, verbose=0,
)
if GPU_COUNT > 0:
    cb_params_base["devices"] = CATBOOST_DEVICES
cb_params = [
    dict(learning_rate=0.020, random_seed=7, **cb_params_base),
]

xgb_params_base = dict(
    n_estimators=8000, max_depth=8,  # v39=10000, v40=8000
    min_child_weight=6,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=3.0,
    reg_alpha=0.05,
    objective='reg:squarederror', eval_metric='rmse',
    tree_method='hist',
    device='cuda' if GPU_COUNT > 0 else 'cpu',
    early_stopping_rounds=400,
    verbosity=0,
)
xgb_params = [
    dict(learning_rate=0.030, random_state=123, **xgb_params_base),
]

oof_preds = {}; test_preds = {}; overall_scores = {}; fold_scores = {}

NON_CAUSAL_PREFIXES = (
    'beam_', 'sc', 'hyb_', 'dtw_', 'tddtw', 'tdbc', 'tdsc', 'sig_',
    'glead', 'glag', 'gr', 'tda', 'tdpf', 'pf_z', 'pf_vs_z',
)
NON_CAUSAL_EXACT = {
    'beam_vs_spatial', 'eval_len', 'frac', 'frac2', 'sqrt_frac',
}

def causal_feature_names(feature_names):
    return [
        c for c in feature_names
        if c not in NON_CAUSAL_EXACT
        and not any(c.startswith(prefix) for prefix in NON_CAUSAL_PREFIXES)
    ]

def diagnostic_lightgbm_cv(name, feature_names, cv_splits, spatial_overlays=None):
    params = dict(lgb_params_base)
    params.update(learning_rate=0.04, seed=SEED)
    oof = np.zeros(len(train_df), np.float32)
    for fold, (tr_idx, va_idx) in enumerate(cv_splits):
        if spatial_overlays is None:
            X_fold = train_df[feature_names]
        else:
            X_fold = overlay_spatial_features(
                train_df, spatial_overlays[fold], features)[feature_names]
        d_tr = lgb.Dataset(X_fold.iloc[tr_idx], label=y.iloc[tr_idx])
        d_va = lgb.Dataset(X_fold.iloc[va_idx], label=y.iloc[va_idx], reference=d_tr)
        model = lgb.train(
            params, d_tr, valid_sets=[d_va],
            num_boost_round=CFG.diagnostic_estimators,
            callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va_idx] = model.predict(
            X_fold.iloc[va_idx], num_iteration=model.best_iteration).astype(np.float32)
    score = root_mean_squared_error(y, oof)
    _dbg(f"DIAGNOSTIC {name}: OOF RMSE={score:.4f}")
    return oof, score

def run_holdout_ablation(name, feature_names):
    tr_idx, va_idx = splits[0]
    X_fold = overlay_spatial_features(
        train_df, fold_spatial_train[0], features)[feature_names]
    params = dict(lgb_params_base)
    params.update(learning_rate=0.04, seed=SEED)
    d_tr = lgb.Dataset(X_fold.iloc[tr_idx], label=y.iloc[tr_idx])
    d_va = lgb.Dataset(X_fold.iloc[va_idx], label=y.iloc[va_idx], reference=d_tr)
    model = lgb.train(
        params, d_tr, valid_sets=[d_va],
        num_boost_round=CFG.diagnostic_estimators,
        callbacks=[lgb.early_stopping(200, verbose=False)])
    pred = model.predict(X_fold.iloc[va_idx], num_iteration=model.best_iteration)
    score = root_mean_squared_error(y.iloc[va_idx], pred)
    _dbg(f"ABLATION fold0 {name}: RMSE={score:.4f} features={len(feature_names)}")
    return score

diagnostic_scores = {}
if CFG.run_diagnostics:
    causal_features = causal_feature_names(features)
    causal_oof, diagnostic_scores['causal-group-cv'] = diagnostic_lightgbm_cv(
        'causal group CV', causal_features, splits, fold_spatial_train)
    for lo, hi in ((0.0, 0.33), (0.33, 0.67), (0.67, 1.01)):
        mask = (train_df['frac'].values >= lo) & (train_df['frac'].values < hi)
        score = root_mean_squared_error(y.values[mask], causal_oof[mask])
        _dbg(f"  causal prefix frac [{lo:.2f}, {hi:.2f}): RMSE={score:.4f}")

    no_spatial_features = [c for c in features if c not in SPATIAL_FEATURE_COLUMNS]
    _, diagnostic_scores['spatial-block-cv'] = diagnostic_lightgbm_cv(
        'spatial-block CV', features, spatial_splits, spatial_diag_overlays)
    _, diagnostic_scores['spatial-block-no-spatial'] = diagnostic_lightgbm_cv(
        'spatial-block CV (spatial features excluded)',
        no_spatial_features, spatial_splits)

    ablation_sets = {
        'full': features,
        'no-stratigraphic-position': [
            c for c in features
            if not (c.startswith('fmd_') or c.startswith('thk_')
                    or c in {'pos_strat', 'closest_fm_idx', 'closest_fm_dist'})
        ],
        'no-dtw': [
            c for c in features
            if not (c.startswith('dtw_') or c.startswith('tddtw')
                    or c in {'sig_std', 'sig_mean_d'})
        ],
        'no-spatial': no_spatial_features,
    }
    for ablation_name, ablation_features in ablation_sets.items():
        diagnostic_scores[f'ablation-{ablation_name}'] = run_holdout_ablation(
            ablation_name, ablation_features)
    del spatial_diag_overlays

In [ ]:
def train_lightgbm(params, name):
    params = dict(params); n_est = params.pop('n_estimators')
    oof = np.zeros(len(train_df), np.float32); tst = np.zeros(len(test_df), np.float32)
    fold_sc = []
    _dbg(f"LGB[{name}] n_est={n_est}, lr={params['learning_rate']}, seed={params['seed']}")
    for fi, (tr_idx, va_idx) in enumerate(splits):
        X_fold = overlay_spatial_features(train_df, fold_spatial_train[fi], features)
        X_test_fold = overlay_spatial_features(test_df, fold_spatial_test[fi], features)
        d_tr = lgb.Dataset(X_fold.iloc[tr_idx], label=y.iloc[tr_idx])
        d_va = lgb.Dataset(X_fold.iloc[va_idx], label=y.iloc[va_idx], reference=d_tr)
        m = lgb.train(params, d_tr, valid_sets=[d_va], num_boost_round=n_est,
                      callbacks=[lgb.early_stopping(400, verbose=False)])  # was 250
        oof[va_idx] = m.predict(
            X_fold.iloc[va_idx], num_iteration=m.best_iteration).astype(np.float32)
        tst += m.predict(
            X_test_fold, num_iteration=m.best_iteration).astype(np.float32) / N_SPLITS
        sc = root_mean_squared_error(y.iloc[va_idx], oof[va_idx])
        fold_sc.append(sc)
        _dbg(f"  LGB[{name}] fold {fi}: RMSE={sc:.4f} (best_iter={m.best_iteration})")
    ov = root_mean_squared_error(y, oof)
    _dbg(f"LGB[{name}] overall: {ov:.4f}")
    return oof, tst, ov, fold_sc

def train_catboost(params, name):
    params = dict(params)
    oof = np.zeros(len(train_df), np.float32); tst = np.zeros(len(test_df), np.float32)
    fold_sc = []
    _dbg(f"CB[{name}] lr={params['learning_rate']}, seed={params['random_seed']}")
    for fi, (tr_idx, va_idx) in enumerate(splits):
        X_fold = overlay_spatial_features(train_df, fold_spatial_train[fi], features)
        X_test_fold = overlay_spatial_features(test_df, fold_spatial_test[fi], features)
        m = CatBoostRegressor(**params)
        m.fit(Pool(X_fold.iloc[tr_idx].values, label=y.iloc[tr_idx].values),
              eval_set=Pool(X_fold.iloc[va_idx].values, label=y.iloc[va_idx].values),
              use_best_model=True)
        oof[va_idx] = m.predict(X_fold.iloc[va_idx]).astype(np.float32)
        tst += m.predict(X_test_fold).astype(np.float32) / N_SPLITS
        sc = root_mean_squared_error(y.iloc[va_idx], oof[va_idx])
        fold_sc.append(sc)
        _dbg(f"  CB[{name}] fold {fi}: RMSE={sc:.4f} (best_iter={m.tree_count_})")
    ov = root_mean_squared_error(y, oof)
    _dbg(f"CB[{name}] overall: {ov:.4f}")
    return oof, tst, ov, fold_sc

def train_xgboost(params, name):
    params = dict(params)
    oof = np.zeros(len(train_df), np.float32); tst = np.zeros(len(test_df), np.float32)
    fold_sc = []
    _dbg(f"XGB[{name}] lr={params['learning_rate']}, seed={params['random_state']}")
    for fi, (tr_idx, va_idx) in enumerate(splits):
        X_fold = overlay_spatial_features(train_df, fold_spatial_train[fi], features)
        X_test_fold = overlay_spatial_features(test_df, fold_spatial_test[fi], features)
        m = xgb.XGBRegressor(**params)
        m.fit(X_fold.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_fold.iloc[va_idx], y.iloc[va_idx])], verbose=False)
        oof[va_idx] = m.predict(X_fold.iloc[va_idx]).astype(np.float32)
        tst += m.predict(X_test_fold).astype(np.float32) / N_SPLITS
        sc = root_mean_squared_error(y.iloc[va_idx], oof[va_idx])
        fold_sc.append(sc)
        _dbg(f"  XGB[{name}] fold {fi}: RMSE={sc:.4f} (best_iter={m.best_iteration})")
    ov = root_mean_squared_error(y, oof)
    _dbg(f"XGB[{name}] overall: {ov:.4f}")
    return oof, tst, ov, fold_sc

for i in range(len(lgb_params)):
    n = f"lightgbm-{i+1}"
    oof_, tst_, ov_, fs_ = train_lightgbm(lgb_params[i], n)
    oof_preds[n] = oof_; test_preds[n] = tst_; overall_scores[n] = ov_; fold_scores[n] = fs_

for i in range(len(cb_params)):
    n = f"catboost-{i+1}"
    oof_, tst_, ov_, fs_ = train_catboost(cb_params[i], n)
    oof_preds[n] = oof_; test_preds[n] = tst_; overall_scores[n] = ov_; fold_scores[n] = fs_

for i in range(len(xgb_params)):
    n = f"xgboost-{i+1}"
    oof_, tst_, ov_, fs_ = train_xgboost(xgb_params[i], n)
    oof_preds[n] = oof_; test_preds[n] = tst_; overall_scores[n] = ov_; fold_scores[n] = fs_

## 10. Cross-fitted blend

Blend weights are fitted on four folds and applied to the untouched fifth
fold. Test predictions average the five independently fitted weight sets.

In [ ]:
_dbg("Cross-fitted hill climber")
oof_preds_df = pd.DataFrame(oof_preds)
test_preds_df = pd.DataFrame(test_preds)
_dbg(f"Models in HC: {list(oof_preds_df.columns)}")

def proper_hill_climb_v2(oof_df, test_df_blend, y_arr,
                         precision=0.0005, score_dp=5, max_iters=5000,
                         min_iters_per_step=10, verbose=True):
    cols = list(oof_df.columns)
    oof_arr = {c: oof_df[c].values.astype(np.float64) for c in cols}
    tst_arr = {c: test_df_blend[c].values.astype(np.float64) for c in cols}
    sc_single = {c: root_mean_squared_error(y_arr, oof_arr[c]) for c in cols}
    best_m = min(sc_single, key=sc_single.get)
    w = {c: 0.0 for c in cols}; w[best_m] = 1.0
    blend = oof_arr[best_m].copy(); total = 1.0; best_score = sc_single[best_m]
    _dbg(f"  climber: start {best_m} @ {best_score:.5f}")
    step_schedule = [s for s in (0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 0.0005)
                     if s >= precision]
    iter_n = 0
    for step_i, step in enumerate(step_schedule):
        step_iters = 0; no_improvement_streak = 0
        while iter_n < max_iters:
            new_total = total + step
            best_pick = None; best_trial_score = best_score; best_trial_blend = None
            for m in cols:
                trial_blend = (blend * total + step * oof_arr[m]) / new_total
                trial_score = root_mean_squared_error(y_arr, trial_blend)
                if (round(trial_score, score_dp) < round(best_trial_score, score_dp)
                        and trial_score < best_trial_score):
                    best_trial_score = trial_score; best_pick = m; best_trial_blend = trial_blend
            if best_pick is None:
                no_improvement_streak += 1
                if step_iters >= min_iters_per_step or step_i == 0: break
                if no_improvement_streak >= 3: break
                continue
            w[best_pick] += step
            blend = best_trial_blend; total = new_total; best_score = best_trial_score
            iter_n += 1; step_iters += 1; no_improvement_streak = 0
            if verbose and (iter_n <= 30 or iter_n % 20 == 0):
                _dbg(f"  it {iter_n}: +{step:.4f} {best_pick} w={w[best_pick]:.4f} -> {best_score:.5f}")
        if iter_n >= max_iters:
            _dbg(f"  climber: hit max_iters={max_iters}"); break
    _dbg(f"  climber: {iter_n} total iters, raw total weight={total:.3f}")
    norm_w = {k: v / total for k, v in w.items()}
    hc_oof = blend.astype(np.float32)
    hc_tst = np.zeros(len(test_df_blend), np.float64)
    for k, v in norm_w.items():
        if v > 0: hc_tst += v * tst_arr[k]
    return hc_oof, hc_tst.astype(np.float32), float(best_score), norm_w

hc_oof_preds = np.zeros(len(train_df), np.float32)
hc_test_preds = np.zeros(len(test_df), np.float64)
hc_weight_folds = []
hc_fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(splits):
    _, fold_test, _, fold_weights = proper_hill_climb_v2(
        oof_preds_df.iloc[tr_idx].reset_index(drop=True),
        test_preds_df,
        y.iloc[tr_idx].to_numpy(),
        precision=0.005, score_dp=4, max_iters=500,
        min_iters_per_step=5, verbose=False)
    fold_val = np.zeros(len(va_idx), np.float64)
    for model_name, weight in fold_weights.items():
        fold_val += weight * oof_preds_df[model_name].values[va_idx]
    hc_oof_preds[va_idx] = fold_val.astype(np.float32)
    hc_test_preds += fold_test / N_SPLITS
    fold_score = root_mean_squared_error(y.iloc[va_idx], fold_val)
    hc_fold_scores.append(fold_score)
    hc_weight_folds.append(fold_weights)
    _dbg(f"  cross-fit blend fold {fold}: RMSE={fold_score:.4f}")
hc_test_preds = hc_test_preds.astype(np.float32)
hc_score = root_mean_squared_error(y, hc_oof_preds)
hc_weights = {
    model_name: float(np.mean([w[model_name] for w in hc_weight_folds]))
    for model_name in oof_preds_df.columns
}
_dbg(f"Cross-fitted blend final: {hc_score:.4f}")
_w_str = ", ".join("%s: %.4f" % (k, v) for k, v in
                   sorted(hc_weights.items(), key=lambda kv: -kv[1]) if v > 1e-4)
_dbg("Mean blend weights: { " + _w_str + " }")
fold_scores["hill-climbing"] = hc_fold_scores
overall_scores["hill-climbing"] = hc_score

## 11. Cross-fitted post-processing and smoothing

Parameters are fitted on four folds, applied to the untouched fifth fold,
and include the same smoothing step used for test inference.

In [ ]:
def apply_pp(df, md, pd_, alpha, tau, w_pf):
    d = md * (1.0 - w_pf) + pd_ * w_pf
    if tau:
        d *= (1.0 - np.exp(-np.maximum(df['md_since'].values, 0.0) / tau))
    return d * alpha

def sg_smooth_values(df, values, sg_p=3):
    """Apply the exact submission smoother and return an aligned array."""
    out = pd.Series(np.asarray(values, np.float64), index=df.index)
    for _, group in df.groupby('well', sort=False):
        v = out.loc[group.index].to_numpy()
        n = len(v)
        sg_w = max(9, min(31, (n // 10) | 1))
        wl = min(sg_w, n)
        if wl % 2 == 0:
            wl -= 1
        if wl >= sg_p + 2:
            v = savgol_filter(v, wl, sg_p)
        out.loc[group.index] = v
    return out.to_numpy(np.float32)

def postprocess_full_tvt(df, model_delta, pf_delta, params):
    base_local = df['last_known_tvt'].to_numpy()
    pred = base_local + apply_pp(
        df, model_delta, pf_delta,
        params['alpha'], params['tau'], params['w_pf'])
    return sg_smooth_values(df, pred)

_dbg(f"Cross-fitted Optuna PP ({CFG.pp_trials} trials per fold, smoothing included)")
base = train_df['last_known_tvt'].values
ytrue_full = y.values + base
pf_oof = train_df['pf_ancc'].values - base

optuna.logging.set_verbosity(optuna.logging.WARNING)
pf_test = test_df['pf_ancc'].values - test_df['last_known_tvt'].values
pp_oof_full = np.zeros(len(train_df), np.float32)
pp_test_full = np.zeros(len(test_df), np.float64)
pp_params_folds = []
pp_fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(splits):
    tr_df = train_df.iloc[tr_idx].copy()
    va_df = train_df.iloc[va_idx].copy()
    fold_weights = hc_weight_folds[fold]
    fold_blend_train = np.zeros(len(tr_idx), np.float64)
    fold_blend_valid = np.zeros(len(va_idx), np.float64)
    fold_blend_test = np.zeros(len(test_df), np.float64)
    for model_name, weight in fold_weights.items():
        fold_blend_train += weight * oof_preds_df[model_name].values[tr_idx]
        fold_blend_valid += weight * oof_preds_df[model_name].values[va_idx]
        fold_blend_test += weight * test_preds_df[model_name].values

    def _pp_objective(trial):
        params = {
            'alpha': trial.suggest_float('alpha', 0.85, 1.35, step=0.005),
            'tau': trial.suggest_int('tau', 20, 400, step=5),
            'w_pf': trial.suggest_float('w_pf', 0.0, 0.35, step=0.01),
        }
        pred = postprocess_full_tvt(
            tr_df, fold_blend_train, pf_oof[tr_idx], params)
        return root_mean_squared_error(ytrue_full[tr_idx], pred)

    sampler = optuna.samplers.TPESampler(
        seed=SEED + fold, n_startup_trials=min(25, CFG.pp_trials // 3))
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(_pp_objective, n_trials=CFG.pp_trials, n_jobs=1)
    params = dict(study.best_params)
    pp_params_folds.append(params)
    pp_oof_full[va_idx] = postprocess_full_tvt(
        va_df, fold_blend_valid, pf_oof[va_idx], params)
    pp_test_full += postprocess_full_tvt(
        test_df, fold_blend_test, pf_test, params) / N_SPLITS
    fold_score = root_mean_squared_error(ytrue_full[va_idx], pp_oof_full[va_idx])
    pp_fold_scores.append(fold_score)
    _dbg(f"  PP fold {fold}: RMSE={fold_score:.4f} params={params}")

pp_oof_rmse = root_mean_squared_error(ytrue_full, pp_oof_full)
mean_pp_params = {
    'alpha': float(np.mean([p['alpha'] for p in pp_params_folds])),
    'tau': float(np.mean([p['tau'] for p in pp_params_folds])),
    'w_pf': float(np.mean([p['w_pf'] for p in pp_params_folds])),
}
_dbg(f"Cross-fitted PP+smoothing TVT RMSE={pp_oof_rmse:.4f}")

overall_scores["hill-climbing (pp)"] = pp_oof_rmse
fold_scores["hill-climbing (pp)"] = pp_fold_scores

## 12. Test inference + adaptive SavGol smoothing

In [ ]:
_dbg("Building test predictions")
test_df2 = test_df.copy()
test_df2['pred'] = pp_test_full.astype(np.float32)

## 13. Submission

In [ ]:
_dbg("Writing submission.csv")
sample_path = CFG.dataset_path / "sample_submission.csv"
if sample_path.exists():
    sample_sub = pd.read_csv(sample_path)
else:
    sample_sub = pd.DataFrame({'id': test_df2['id']})

if 'id' not in sample_sub.columns:
    raise RuntimeError("Sample submission is missing the id column")
if sample_sub['id'].isna().any() or sample_sub['id'].duplicated().any():
    raise RuntimeError("Sample submission IDs must be non-null and unique")

sample_ids = pd.Index(sample_sub['id'])
prediction_ids = pd.Index(test_df2['id'])
missing_prediction_ids = sample_ids.difference(prediction_ids)
extra_prediction_ids = prediction_ids.difference(sample_ids)
if len(missing_prediction_ids) or len(extra_prediction_ids):
    raise RuntimeError(
        "Prediction IDs do not match sample submission IDs: "
        f"missing={missing_prediction_ids[:10].tolist()}, "
        f"extra={extra_prediction_ids[:10].tolist()}")

sub = sample_sub[['id']].merge(
    test_df2[['id', 'pred']].rename(columns={'pred': 'tvt'}),
    on='id', how='left')
n_null = int(sub['tvt'].isnull().sum())
if n_null:
    missing_ids = sub.loc[sub['tvt'].isnull(), 'id'].head(10).tolist()
    raise RuntimeError(
        f"Submission is missing predictions for {n_null} rows; examples={missing_ids}")
if sub['id'].duplicated().any():
    duplicate_ids = sub.loc[sub['id'].duplicated(), 'id'].head(10).tolist()
    raise RuntimeError(f"Submission contains duplicate IDs: {duplicate_ids}")
if not np.isfinite(sub['tvt'].to_numpy(np.float64)).all():
    raise RuntimeError("Submission contains non-finite predictions")

out_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
sub[['id', 'tvt']].to_csv(out_path, index=False)

verify = pd.read_csv(out_path)
if list(verify.columns) != ['id', 'tvt']:
    raise RuntimeError(f"Unexpected submission columns: {list(verify.columns)}")
if len(verify) != len(sample_sub):
    raise RuntimeError(
        f"Submission row count mismatch: {len(verify)} vs {len(sample_sub)}")
if not np.isfinite(verify['tvt'].to_numpy(np.float64)).all():
    raise RuntimeError("Written submission contains non-finite predictions")
_dbg(f"submission.csv verified: {len(verify):,d} rows, "
     f"tvt range [{verify['tvt'].min():.2f}, {verify['tvt'].max():.2f}]")

## 14. Summary

In [ ]:
_dbg("=" * 60)
_dbg("MODEL SUMMARY")
_dbg("=" * 60)
for k in sorted(overall_scores.keys(), key=lambda x: overall_scores[x]):
    sc = overall_scores[k]
    w = hc_weights.get(k, 0.0) if isinstance(hc_weights, dict) else 0.0
    flag = ""
    if k == "hill-climbing": flag = "  <- HC blend"
    elif k == "hill-climbing (pp)": flag = "  <- HC+postproc (final)"
    elif w > 1e-4: flag = f"  (hc_weight={w:.3f})"
    _dbg(f"  {k:32s}  OOF={sc:.4f}{flag}")
_dbg("=" * 60)
_dbg(f"FINAL post-proc OOF RMSE: {pp_oof_rmse:.4f}")
_dbg(f"Mean cross-fit PP params: {mean_pp_params}")
for name, score in diagnostic_scores.items():
    _dbg(f"DIAGNOSTIC {name:28s} {score:.4f}")
_dbg(f"PIPELINE COMPLETE total={int(time.time()-_t0)}s ({(time.time()-_t0)/60:.1f}m)")